In [8]:
import os
import json
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm
from mistralai.client import Mistral
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
MISTRAL_API_KEY = "YOUR_API_KEY"

client = Mistral(api_key=MISTRAL_API_KEY)
MODEL_NAME = "open-mistral-nemo"

In [10]:
def create_batch_system_prompt(golden_df, n_shots=10):
    # Lấy phân phối ngẫu nhiên
    few_shot_df, _ = train_test_split(golden_df, train_size=n_shots, stratify=golden_df['sarcasm'], random_state=42)
    
    prompt = """Bạn là chuyên gia ngôn ngữ học và tâm lý học mạng xã hội Việt Nam. 
Nhiệm vụ: Phân loại danh sách bình luận (Batch) dựa trên Sarcasm và Sentiment.

HƯỚNG DẪN TƯ DUY (CHAIN OF THOUGHT):
Để phân loại chính xác, bạn BẮT BUỘC phải viết 1 câu phân tích ngắn gọn vào trường "reasoning" trước khi chốt nhãn.
1. Tìm sự mâu thuẫn: Lời nói bề mặt có trái ngược với hoàn cảnh/thực tế không? (VD: Khen ngợi một hành động tồi tệ).
2. Dùng tiếng lóng: Bình luận có dùng thành ngữ, icon hoặc từ lóng để nói bóng gió, châm biếm mục tiêu không?
3. LUẬT SENTIMENT: Đánh giá cảm xúc THỰC SỰ mà người nói muốn truyền tải. Lưu ý: Mỉa mai (Sarcasm) hầu như luôn luôn là cảm xúc Tiêu cực (Negative), dù bề mặt câu chữ có vẻ khen ngợi hoặc trung lập.

NHÃN CẦN PHÂN LOẠI:
- Sarcasm: "Sarcastic" HOẶC "Non-Sarcastic".
- Sentiment: "Positive", "Negative", HOẶC "Neutral".

BẮT BUỘC TRẢ VỀ JSON THEO CẤU TRÚC SAU:
{
  "results": [
    {
      "id": "id_câu", 
      "reasoning": "Giải thích ngắn gọn mâu thuẫn hoặc ý đồ...", 
      "sarcasm": "...", 
      "sentiment": "..."
    }
  ]
}

CÁC VÍ DỤ CHUẨN ĐỂ HỌC THEO:
[Input Batch]
"""
    # Nạp đầu vào ví dụ
    for idx, row in few_shot_df.head(4).iterrows(): 
        prompt += f"ID: {idx} | Context: {str(row['video_core_content']).replace(chr(10), ' ')[:150]}... | Comment: {row['comment']} | Commonsense: {row.get('commonsense', '')}\n"
        
    prompt += "\n[Output JSON Mẫu]\n{"
    prompt += '\n  "results": ['
    # Ép nó học cách reasoning
    for idx, row in few_shot_df.head(4).iterrows():
        # Dùng một mẫu reasoning chung chung để nó bắt chước format
        reasoning_text = "Phân tích ngữ cảnh và từ ngữ cho thấy ý đồ thực sự của người viết."
        if row["sarcasm"] == "Sarcastic":
            reasoning_text = "Có sự mâu thuẫn giữa lời nói và thực tế, mang ý đồ châm biếm."
            
        prompt += f'\n    {{"id": "{idx}", "reasoning": "{reasoning_text}", "sarcasm": "{row["sarcasm"]}", "sentiment": "{row["sentiment"]}"}},'
        
    prompt = prompt.rstrip(',') + '\n  ]\n}'
    prompt += "\n\nBÂY GIỜ HÃY SUY LUẬN VÀ PHÂN LOẠI BATCH BÊN DƯỚI:"
    return prompt, few_shot_df.index.tolist()

In [11]:
def call_mistral_batch(system_prompt, batch_data):
    user_input = "[Input Batch]\n"
    for item in batch_data:
        user_input += f"ID: {item['id']}\nContext: {item['context']}\nComment: {item['comment']}\nCommonsense: {item['commonsense']}\n\n"
    
    for attempt in range(5):
        try:
            # Cú pháp gọi hàm của Mistral SDK
            response = client.chat.complete(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                temperature=0.1, 
                response_format={"type": "json_object"} # Bật chế độ ép JSON
            )
            
            content = response.choices[0].message.content
            
            # Xóa các ký tự ẩn (nếu có) để tránh lỗi parse JSON
            import re
            content = re.sub(r'[\x00-\x1f]', '', content)
            
            result_json = json.loads(content)
            return result_json.get("results", [])
            
        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "rate limit" in error_msg:
                wait = 10 * (attempt + 1)
                tqdm.write(f"⏳ Mistral Rate Limit. Tự động chờ {wait}s...")
                time.sleep(wait)
            else:
                tqdm.write(f"❌ Lỗi API hoặc Parse: {e}")
                return None
    return None

In [12]:
def normalize_sarcasm(val):
    val = str(val).lower().strip()
    if 'không' in val or 'non' in val: return 'Non-Sarcastic'
    if 'có' in val or 'mỉa' in val or 'sarcastic' in val: return 'Sarcastic'
    return val

def normalize_sentiment(val):
    val = str(val).lower().strip()
    if 'tích' in val or 'pos' in val: return 'Positive'
    if 'tiêu' in val or 'neg' in val: return 'Negative'
    if 'trung' in val or 'neu' in val: return 'Neutral'
    return val

In [13]:
def plot_confusion_matrix(y_true, y_pred, labels, title, filename=None):
    """
    Hàm vẽ Confusion Matrix chuẩn đẹp mắt.
    """
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(7, 5))
    
    # Vẽ heatmap với màu xanh dương (Blues)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels,
                annot_kws={"size": 14}) # Phóng to số cho dễ nhìn
    
    plt.title(title, fontsize=16, pad=15)
    plt.ylabel('Nhãn Thực Tế (True Label)', fontsize=12)
    plt.xlabel('Nhãn Dự Đoán (Predicted)', fontsize=12)
    plt.tight_layout()
    
    if filename:
        plt.savefig(filename, dpi=300)
        print(f"📸 Đã lưu biểu đồ vào: {filename}")
    
    plt.show() # Hiển thị lên màn hình

In [14]:
if __name__ == "__main__":
    RUN_FULL_DATASET = True 
    BATCH_SIZE = 5 
    
    print("⏳ Đang chuẩn bị dữ liệu...")
    golden_file = "/home/check/DATA/university/yr3, hk2/IE403 - Khai thác dữ liệu truyền thông xã hội/Data/silver/label/labeled/labeled_data.csv"
    df_golden = pd.read_csv(golden_file)
    
    # Ép chuẩn nhãn cho file gốc
    df_golden['sarcasm'] = df_golden['sarcasm'].apply(normalize_sarcasm)
    df_golden['sentiment'] = df_golden['sentiment'].apply(normalize_sentiment)

    # Khởi tạo System Prompt
    system_prompt, few_shot_indices = create_batch_system_prompt(df_golden, n_shots=12)

    # PHASE 1: ĐÁNH GIÁ 

    if not RUN_FULL_DATASET:
        print("\n" + "="*50)
        print(f"PHASE 1: EVALUATION (BATCH={BATCH_SIZE})")
        print("="*50)
        
        df_test = df_golden.drop(few_shot_indices).copy()
        df_test['pred_sarcasm'] = 'ERROR'
        df_test['pred_sentiment'] = 'ERROR'
        
        pending_idx = df_test.index.tolist()
        
        for i in tqdm(range(0, len(pending_idx), BATCH_SIZE), desc="Predicting"):
            batch_indices = pending_idx[i:i+BATCH_SIZE]
            batch_data = []
            for idx in batch_indices:
                row = df_test.loc[idx]
                batch_data.append({
                    "id": str(idx),
                    "context": str(row['video_core_content']).replace("\n", " ")[:150],
                    "comment": str(row['comment']).replace("\n", " "),
                    "commonsense": str(row.get('commonsense', '')).replace("\n", " ")
                })
                
            results = call_mistral_batch(system_prompt, batch_data) 
            
            if results:
                for res in results:
                    row_id = int(res.get("id", -1))
                    if row_id in df_test.index:
                        df_test.at[row_id, 'pred_sarcasm'] = normalize_sarcasm(res.get("sarcasm", "ERROR"))
                        df_test.at[row_id, 'pred_sentiment'] = normalize_sentiment(res.get("sentiment", "ERROR"))
            
            time.sleep(3) 
            
        df_valid = df_test[(df_test['pred_sarcasm'] != 'error') & (df_test['pred_sentiment'] != 'error')]
        
        if len(df_valid) == 0:
            print("❌ TẤT CẢ ĐỀU LỖI.")
        else:
            print("\n" + "="*40)
            print("🏆 BÁO CÁO KẾT QUẢ SARCASM")
            print("="*40)
            print(classification_report(df_valid['sarcasm'], df_valid['pred_sarcasm']))
            print(f"🔥 Sarcasm F1-Macro: {f1_score(df_valid['sarcasm'], df_valid['pred_sarcasm'], average='macro'):.4f}")
            
            sarcasm_labels = ['Non-Sarcastic', 'Sarcastic']
            plot_confusion_matrix(df_valid['sarcasm'], df_valid['pred_sarcasm'], labels=sarcasm_labels, title='Confusion Matrix - Sarcasm', filename='cm_sarcasm.png')
            
            print("\n" + "="*40)
            print("🏆 BÁO CÁO KẾT QUẢ SENTIMENT")
            print("="*40)
            print(classification_report(df_valid['sentiment'], df_valid['pred_sentiment']))
            print(f"🔥 Sentiment F1-Macro: {f1_score(df_valid['sentiment'], df_valid['pred_sentiment'], average='macro'):.4f}")
            
            sentiment_labels = ['Negative', 'Neutral', 'Positive']
            plot_confusion_matrix(df_valid['sentiment'], df_valid['pred_sentiment'], labels=sentiment_labels, title='Confusion Matrix - Sentiment', filename='cm_sentiment.png')


    # PHASE 2: AUTO-LABELING FULL DATA 

    else:
        print("\n" + "="*50)
        print(f"PHASE 2: AUTO-LABELING FULL DATA (BATCH={BATCH_SIZE})")
        print("="*50)
        
        full_data_file = "silver1.csv"
        output_file = "silver1_labeled_final.csv"
        
        if os.path.exists(output_file):
            print("🔄 Tìm thấy file cũ, tiếp tục tiến trình chạy...")
            df_full = pd.read_csv(output_file)
        else:
            df_full = pd.read_csv(full_data_file)
            df_full['pred_sarcasm'] = None
            df_full['pred_sentiment'] = None
            # Thêm cột lưu lại reasoning để sau này bạn có thể đọc xem AI phân tích có đúng ý không
            df_full['reasoning'] = None 
            
        # Tìm các dòng chưa được label hoặc bị lỗi
        pending_idx = df_full[df_full['pred_sarcasm'].isna() | (df_full['pred_sarcasm'] == 'error')].index.tolist()
        print(f"🎯 Cần xử lý: {len(pending_idx)} dòng (Khoảng {len(pending_idx)//BATCH_SIZE + 1} batches).")
        
        for i in tqdm(range(0, len(pending_idx), BATCH_SIZE), desc="Auto-Labeling"):
            batch_indices = pending_idx[i:i+BATCH_SIZE]
            
            batch_data = []
            for idx in batch_indices:
                row = df_full.loc[idx]
                batch_data.append({
                    "id": str(idx),
                    "context": str(row['video_core_content']).replace("\n", " ")[:150],
                    "comment": str(row['comment']).replace("\n", " "),
                    "commonsense": str(row.get('commonsense', '')).replace("\n", " ")
                })
                
            results = call_mistral_batch(system_prompt, batch_data)
            
            if results:
                for res in results:
                    row_id = int(res.get("id", -1))
                    if row_id in df_full.index:
                        # Ép chuẩn nhãn trước khi lưu để data 8000 dòng sạch sẽ 100%
                        df_full.at[row_id, 'pred_sarcasm'] = normalize_sarcasm(res.get("sarcasm", "ERROR"))
                        df_full.at[row_id, 'pred_sentiment'] = normalize_sentiment(res.get("sentiment", "ERROR"))
                        # Lưu cả lời giải thích
                        df_full.at[row_id, 'reasoning'] = res.get("reasoning", "")
            
            # Lưu checkpoint sau mỗi Batch
            df_full.to_csv(output_file, index=False, encoding='utf-8-sig')
            
            # Ngủ 3 giây để không bị đụng Rate Limit
            time.sleep(3) 
            
        print(f"✅ Hoàn thành 100%! Data đã lưu tại {output_file}")

⏳ Đang chuẩn bị dữ liệu...

PHASE 2: AUTO-LABELING FULL DATA (BATCH=5)
🔄 Tìm thấy file cũ, tiếp tục tiến trình chạy...
🎯 Cần xử lý: 5 dòng (Khoảng 2 batches).


Auto-Labeling: 100%|██████████| 1/1 [00:09<00:00,  9.78s/it]

✅ Hoàn thành 100%! Data đã lưu tại silver1_labeled_final.csv
